# Tool check: crnt

Runs each tool of this capability **directly, without the agent**, on the inputs you give. One section per tool: run the setup, then only the section you need.

In [ ]:
import os, sys, time, json
ROOT = os.path.abspath(os.path.join("..", ".."))      # notebooks/tools/ -> project root
sys.path.insert(0, ROOT)
from dynalyst.orchestration import tools as registry
from IPython.display import Image, display


def arguments(tool):
    """The tool's arguments, read from the tool itself."""
    schema = registry.schema_of(registry.all_tools()[tool])
    print(tool, "-- required:", schema.get("required", []))
    for name, spec in schema.get("properties", {}).items():
        print(f"  {name} ({spec.get('type', 'any')}): {(spec.get('description') or '')[:150]}")


def run_tool(tool, args, pasted=""):
    """Run one tool alone: your ARGS, or the agent's call pasted from its log."""
    if pasted.strip():
        args = json.loads(pasted)
    print(tool, args)
    t = time.time()
    raw = registry.execute(tool, args)
    return show(tool, raw, time.time() - t)


def show(tool, raw, seconds):
    """The result: error first, then each field (long ones shortened), then plots."""
    print(f"{tool} ran in {seconds:.1f} s")
    try:
        data = json.loads(raw)
    except (TypeError, ValueError):
        print(str(raw)[:3000])
        return None
    if isinstance(data, dict):
        if data.get("error"):
            print("ERROR:", data["error"])
        for key, value in data.items():
            text = json.dumps(value, default=str)
            print(f"- {key}: {text[:400]}{' ...' if len(text) > 400 else ''}")
        for key, value in data.items():
            if isinstance(value, str) and value.lower().endswith(".png"):
                path = value if os.path.isabs(value) else os.path.join(ROOT, value)
                if os.path.exists(path):
                    print(key)
                    display(Image(filename=path))
    else:
        print(json.dumps(data, default=str)[:3000])
    return data

TOOLS = ['crnt_bistability', 'draw_reaction_network']

## `crnt_bistability`


In [ ]:
arguments("crnt_bistability")
# Fill in what you need (required: []), OR paste the agent's call from its log
# (the {...} after "SELECT crnt_bistability(") into PASTED.
ARGS = {
    # "model_name_or_path": ...,   # string: SBML / XPP model name or path.
    # "species": ...,   # array: 
    # "odes": ...,   # object: {species: rhs} for typed ODEs.
    # "parameters": ...,   # object: 
    # "initial_conditions": ...,   # object: 
    # "iterations": ...,   # integer: Optimisation starts (default 20).
    # "structure_only": ...,   # boolean: Only the network structure (deficiency, stoichiometric matrix, linkage classes, theorems), no search
    # "semi_diffusive": ...,   # boolean: For a network with no conservation law (open, with inflow/outflow): run crnt4sbml's semi-diffusive a
    # "job_id": ...,   # string: Only the job_id of an earlier call that returned still_running and that the user chose to wait for: 
}
PASTED = ""
result = run_tool("crnt_bistability", ARGS, PASTED)

## `draw_reaction_network`


In [ ]:
arguments("draw_reaction_network")
# Fill in what you need (required: []), OR paste the agent's call from its log
# (the {...} after "SELECT draw_reaction_network(") into PASTED.
ARGS = {
    # "model_name_or_path": ...,   # string: SBML / XPP model name or path.
    # "species": ...,   # array: 
    # "odes": ...,   # object: {species: rhs} for typed ODEs.
    # "parameters": ...,   # object: 
    # "initial_conditions": ...,   # object: 
}
PASTED = ""
result = run_tool("draw_reaction_network", ARGS, PASTED)

## The agent on the same request (optional)

In [ ]:
# Optional: ask the agent the same thing and compare (needs the API key).
QUERY = ""                       # e.g. the request that went wrong
if QUERY.strip():
    from dynalyst.agents.unified_agent import run
    t = time.time()
    rec = run(QUERY)
    print(f"agent: {time.time() - t:.1f} s\n")
    calls = [s for s in rec.steps if s.get("tool") in TOOLS]
    print(f"the agent made {len(calls)} call(s) to these tools:")
    for a in calls:
        print(" ", a.get("tool"), json.dumps(a.get("arguments"), default=str)[:400],
              "|", a.get("status"), (a.get("error") or "")[:200])
    print("\nanswer:\n", rec.final_answer[:2000])

**How to read it**
- The tool fails or gives a wrong result here, with the agent's arguments: the **tool** is the cause.
- The tool works here, but the chat went wrong: the **agent** is the cause (wrong arguments,
  wrong tool, or a wrong reading of the result). Compare with the agent's calls above.
- The tool works here only with different arguments: the agent passed wrong arguments.